# Wave 0: Tool-Call Observability

## Goal

Add the smallest shared observability slice that makes non-LLM retrieval operations queryable in the same backend as `llm_client`.

Wave 0 intentionally does **not** add:
- span hierarchies
- gather tracking
- project-phase instrumentation
- a separate telemetry stack

It only needs to answer:
- how many search calls failed, and why?
- how many fetch calls failed, and why?
- which provider or domain fails most?
- how long do successful vs failed calls take?
- what happened for one retrieval trace?


## Storage Decision

Use a dedicated `tool_calls` table plus matching JSONL output inside `llm_client`.

Why this is the right Wave 0 choice:
- simpler SQL than repurposing `foundation_events`
- avoids forcing open-web retrieval into the heavier foundation-event schema
- keeps the event model small and typed
- leaves room to bridge into foundation events later if needed

This still uses the existing `llm_client` observability backend and lifecycle rules.


## ToolCallResult Contract

Fields:
- `call_id`: stable id shared across started/final records for one operation attempt
- `tool_name`: logical tool surface, initially `open_web_retrieval`
- `operation`: `search`, `fetch`, or `extract`
- `provider`: provider or implementation path (`brave`, `searxng`, `httpx`, `trafilatura`, etc.)
- `target`: compact target identifier (URL or query preview)
- `status`: `started`, `succeeded`, or `failed`
- `started_at`, `ended_at`, `duration_ms`
- `attempt`: 1-based attempt count
- `task`, `trace_id`
- `metrics`: compact JSON dict
- `error_type`, `error_message`

Constraints:
- logging must never break tool execution
- payloads must stay compact
- rows must be queryable by `trace_id`, `task`, `tool_name`, `operation`, `provider`, and `status`


## Example Records

Successful Brave search:

```json
{
  "call_id": "toolcall_abc123",
  "tool_name": "open_web_retrieval",
  "operation": "search",
  "provider": "brave",
  "target": "query:ubi pilot programs",
  "status": "succeeded",
  "attempt": 1,
  "trace_id": "trace-ubi-001",
  "task": "grounded_research.collect",
  "duration_ms": 1180,
  "metrics": {
    "top_k": 10,
    "returned_count": 8,
    "query_sha256": "d3b0..."
  }
}
```

Failed fetch:

```json
{
  "call_id": "toolcall_def456",
  "tool_name": "open_web_retrieval",
  "operation": "fetch",
  "provider": "httpx",
  "target": "https://example.com/article",
  "status": "failed",
  "attempt": 1,
  "trace_id": "trace-ubi-001",
  "task": "grounded_research.collect",
  "duration_ms": 5042,
  "error_type": "FetchError",
  "error_message": "HTTP 403",
  "metrics": {
    "http_status": 403,
    "domain": "example.com",
    "retryable": false
  }
}
```

Fallback fetch path as separate visible calls:

```json
{"call_id": "toolcall_ghi789", "operation": "fetch", "provider": "httpx", "status": "failed"}
{"call_id": "toolcall_jkl012", "operation": "fetch", "provider": "jina_reader", "status": "succeeded"}
```


## Acceptance SQL

Failure counts by provider:

```sql
SELECT operation, provider, status, COUNT(*) AS n
FROM tool_calls
WHERE status = 'failed'
GROUP BY operation, provider, status
ORDER BY n DESC;
```

Slowest fetches:

```sql
SELECT target, provider, duration_ms, error_type
FROM tool_calls
WHERE operation = 'fetch' AND status IN ('succeeded', 'failed')
ORDER BY duration_ms DESC
LIMIT 20;
```

One trace timeline:

```sql
SELECT timestamp, call_id, operation, provider, target, status, duration_ms, error_type
FROM tool_calls
WHERE trace_id = ?
ORDER BY timestamp, id;
```

Failed fetches by domain:

```sql
SELECT json_extract(metrics, '$.domain') AS domain, COUNT(*) AS n
FROM tool_calls
WHERE operation = 'fetch' AND status = 'failed'
GROUP BY domain
ORDER BY n DESC;
```


## Wave 0 Exit Gate

Wave 0 is done only if:
- `llm_client` can persist typed tool-call rows to JSONL and SQLite
- `open_web_retrieval` emits search, fetch, and extract records at the real function boundaries
- a single trace can be reconstructed from the database alone
- we can count failures and compare durations without adding any span/gather model

Only if those records are insufficient do we move to Wave 1.
